# ICS 604: APPLIED DATA SCIENCE

## Non-Linear Regression

- ### Nearest neighbor regression
- ### Step functions
- ### Polynomial regression
---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

In [ ]:
np.random.seed(0)

## Non-Linear Regression

Non-linear regression reflects the reality that many real-world relationships are too complex to be accurately captured by simple straight-line models. While linear regression remains a foundational and widely used technique, especially for its simplicity and interpretability, it often falls short when patterns in the data involve curves, interactions, or more intricate structures. Non-linear approaches allow models to better adapt to these complexities, providing a more flexible framework for capturing underlying trends.

That said, linearity is often “good enough” in many practical situations. Linear models are especially valuable when quickly prototyping solutions or when interpretability is a top priority, such as in early-stage analysis or when communicating results to non-technical stakeholders. They also work well for problems that are not yet well-defined, where introducing additional complexity may not yield meaningful improvements and could instead obscure insights.

Ultimately, the goal is not simply to build more complex models, but to build better ones. As we explore non-linear approaches, it is important to weigh the added precision against the practical benefits it brings. A more sophisticated model may offer improved accuracy, but only if that improvement translates into meaningful leverage — whether in decision-making, prediction quality, or actionable insights.

### Non-Linearity

To illustrate non-linearity, we consider data generated from a quadratic model:

$$
y = 30 -0.3 x +0.005 x^2 + \epsilon
$$

This formulation includes both a linear term and a squared term, allowing the relationship between $x$ and $y$ to curve rather than follow a straight line. As a result, a simple linear regression model would struggle to accurately capture this pattern, since it cannot account for the changing rate of increase or decrease introduced by the $x^2$ component.

In [ ]:
errors = np.random.normal(0, 3, size=200)
x = np.linspace(0, 100, 200)
y = 30 + (-0.3 * x)+ (0.005 * x ** 2) + errors

plt.figure(figsize=(8, 4))
plt.scatter(x, y);

In [ ]:
from scipy.stats import linregress

lm = linregress(x, y)

plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
plt.scatter(x, y, alpha=0.3)
plt.plot(x, lm.intercept + lm.slope * x, color='r', linewidth=4)
plt.title("Data and linear fit", fontsize=16)

Res_vals = []
for (x_i, y_i) in zip(x, y):
    y_hat = lm.intercept + lm.slope * x_i 
    Res_vals.append(y_i - y_hat)

plt.subplot(1, 2, 2)
plt.scatter(x, Res_vals)
plt.title("Residuals", fontsize=16);

In addition to the non-linear structure, the behavior of the residuals further complicates the modeling process. While the true error term $\epsilon$ is generated from a normal distribution (i.e., $\epsilon \sim \mathcal{N}(\mu,\sigma)$), the issue arises after fitting an incorrect model. If we apply a simple linear regression that omits the quadratic term, the model becomes misspecified and fails to capture the curvature in the data. As a result, the residuals — defined as the differences between observed and predicted values — will exhibit systematic patterns rather than purely random noise, and they may no longer appear normally distributed. This apparent violation of the normality assumption is therefore not due to the data-generating process itself, but rather due to the inadequacy of the chosen model.

### How to Determine a Non-Linear Model

When faced with data that clearly exhibits a non-linear pattern, the challenge becomes how to model it effectively without overcomplicating the approach. A fully specified non-linear model may be appropriate, but another useful perspective is to consider how the relationship behaves locally rather than globally. Instead of forcing a single functional form across the entire range of the data, we can examine smaller segments where the structure may be simpler.

Within a sufficiently small region, many non-linear relationships can be well-approximated by a linear trend. This idea is rooted in the notion that smooth curves, when “zoomed in” on, begin to resemble straight lines. By leveraging this property, we can build intuition about the data or even construct models that approximate the overall curve through a series of local linear fits.

For example, if we focus on the region $x \in [35, 45]$, the curvature of the quadratic relationship becomes less pronounced over that narrow interval. In this localized window, a linear model may provide a reasonable approximation of the relationship between $x$ and $y$. This illustrates how non-linear behavior at a global scale can still be understood and approximated through simpler linear methods when viewed locally.

In [ ]:
positions = np.where((x >= 35) & (x <= 45))
x[positions]

In [ ]:
y[positions]

In [ ]:
lm_2 = linregress(x[positions], y[positions])

plt.figure(figsize=(16, 4))

plt.subplot(1, 3, 1)
plt.scatter(x[positions], y[positions], alpha=0.3)
plt.plot(x[positions], lm_2.intercept + lm_2.slope * x[positions], color='r', linewidth=4)
plt.ylim(y.min(), y.max())
plt.xlim(25, 55)
plt.title("Data and linear fit", fontsize=16)

plt.subplot(1, 3, 2)
plt.scatter(x[positions], y[positions], alpha=0.3)
plt.plot(x[positions], lm_2.intercept + lm_2.slope * x[positions], color='r', linewidth=4)
plt.xlim(25, 55)
plt.title("Data and linear fit (zoomed)", fontsize=16)

Res_vals = []
for (x_i, y_i) in zip(x[positions], y[positions]):
    y_hat = lm_2.intercept + lm_2.slope * x_i 
    Res_vals.append(y_i - y_hat)

plt.subplot(1, 3, 3)
plt.scatter(x[positions], Res_vals)
plt.xlim(25, 55)
plt.title("Residuals", fontsize=16);

### Nearest Neighbor Regression

A straightforward way to handle non-linear data is to use a Nearest Neighbor (NN) regression. The idea is simple: to predict the value at a given point, we take the average of nearby observed points. For example, if we want to predict $y$ at $x=40$, we could select the 3 observed points immediately before and after 40, or any fixed number of nearest neighbors — say, 5 points — and compute their mean. This approach effectively smooths the data locally and provides a prediction that reflects nearby behavior rather than forcing a global linear or non-linear fit. In practice, functions like `np.searchsorted(sorted_array, q)` can be used to quickly locate the position of the smallest value in `sorted_array` that is larger than `q`, helping identify neighbors efficiently.

In [ ]:
x[130:150]

In [ ]:
np.searchsorted(x, 70)

In [ ]:
x[np.searchsorted(x, 70)]

In [ ]:
pos = np.searchsorted(x, 70)
print(x[pos: pos+5])
print(x[pos-5: pos])

In [ ]:
plt.figure(figsize=(10, 5))

neighbors = np.arange(pos-5, pos+5)
plt.scatter(x, y, alpha=0.3)
plt.scatter(x[neighbors], y[neighbors], color="red");

In [ ]:
plt.figure(figsize=(10, 5))

reg_curve = []

for i in x[5:-4]:
    pos = np.searchsorted(x, i)
    neighbors = np.arange(pos-5, pos+5)
    reg_curve.append(y[neighbors].mean())

plt.scatter(x, y, alpha=0.3)
plt.plot(x[5:-4], reg_curve, color="red");

#### Problems with Nearest Neighbor Regression

Despite its simplicity, nearest neighbor regression has several limitations. One major issue is that it can be **computationally slow**: for each prediction, distances between the query point and all points in the dataset must be calculated to identify the nearest neighbors. This becomes increasingly expensive as the number of data points grows or as the number of predictors (dimensions) increases. The method is also **sensitive to outliers**, since extreme values among the nearest neighbors can disproportionately affect the mean prediction. Finally, nearest neighbor regression struggles near the **edges of the data**, where there are fewer neighbors to average, making predictions at the extremities unreliable or impossible. These constraints mean that while nearest neighbor regression is intuitive and locally adaptive, it is best suited for smaller, well-sampled datasets.

### Step Functions

One way to address some of the limitations of nearest neighbor regression is to discretize the $x$-axis. By breaking the range of $x$ into bins, we can fit a separate constant value within each bin, effectively smoothing the data locally. This constant is often chosen as the mean of the points in the bin, similar to the approach used in nearest neighbors. By treating the continuous variable as an ordered categorical variable, we create a piecewise-constant approximation of the original relationship. This approach is commonly referred to as a step function, as the fitted values appear as horizontal “steps” across consecutive bins.

In [ ]:
intervals = np.split(np.arange(len(x)), 10)
intervals

In [ ]:
intervals[4]

In [ ]:
x[intervals[4]]

In [ ]:
y[intervals[4]].mean()

In [ ]:
plt.figure(figsize=(10, 5))

plt.scatter(x, y, alpha=0.3)
plt.plot(x[intervals[4]], [y[intervals[4]].mean()] * len(intervals[4]), color="r", linewidth=3);

In [ ]:
plt.figure(figsize=(10, 5))

plt.scatter(x, y, alpha=0.3)

for i in range(len(intervals)):
    plt.plot(x[intervals[i]], [y[intervals[i]].mean()] * len(intervals[i]), color="r", linewidth=3)

#### Shortcomings of Step Functions

Step functions introduce their own challenges. One issue is the **sudden jumps** between adjacent bins, which can be hard to interpret when two points are very close on the $x$-axis but fall into different intervals. Another challenge is the **choice of cutpoints or “knots”**: the results of a step function can vary significantly depending on how many bins are chosen or exactly where the splits occur. For example, using 11, 9, or 13 intervals could produce noticeably different fits. This arbitrary selection of knots can lead to substantial variation in predictions, making the model sensitive to how the discretization is performed.

### Polynomial Regression

An alternative approach to modeling non-linear relationships is polynomial regression, where we extend the linear model by including higher-degree terms of the predictor. Instead of restricting ourselves to a first-degree (linear) relationship, we allow the model to incorporate curvature by adding squared, cubic, or even higher-order terms. This provides greater flexibility in capturing more complex patterns in the data while still using a familiar regression framework.

A standard linear model is a first-degree polynomial of the form:

$$
    y = \beta_0 + \beta_1 x
$$

which assumes a straight-line relationship between $x$ and $y$. In contrast, a third-degree polynomial model includes additional terms such as $x^2$ and $x^3$:

$$ 
    y = \beta_0 + \beta_1 x + \beta_2x^2 + \beta_3x^3 
$$

These higher-order terms allow the model to bend and adapt to changes in the data, making it better suited for capturing non-linear trends while still being linear in the parameters $\beta$.

#### Using Linear Models with Polynomial Features

In practice, implementing polynomial regression — such as with the scikit-learn library — follows a simple two-step process. First, the original input feature $x$ is transformed into higher-degree features (e.g., $x^2$, $x^3$). Then, a standard linear regression model is fit using these transformed features. This allows us to capture non-linear relationships while still relying on well-understood linear modeling techniques.

From an implementation standpoint, this process is still just Ordinary Least Squares (OLS). The polynomial model

$$ 
    y = \beta_0 + \beta_1x + \beta_2x^2 + \beta_3x^3 
$$

can be rewritten by introducing new features:

$$
    y = \beta_0 \cdot 1 + \beta_1 \cdot A + \beta_2 \cdot B + \beta_3 \cdot C
$$

where $A=x$, $B=x^2$, and $C=x^3$. By doing this, we convert the problem into a multivariate linear regression with features $A$, $B$, and $C$. Importantly, the model is still considered *linear* because it remains linear in the coefficients $\beta$, even though it is non-linear in $x$.

To automate this transformation, we can use the `PolynomialFeatures` utility in `scikit-learn`. This tool generates all polynomial combinations of the input features up to a specified degree, making it easy to expand the feature space. The input data is typically provided as a column vector, and the transformation augments it with additional columns corresponding to powers of $x$, including the constant term $x^0=1$.

For example, consider the input vector $x=[1,2,3,4,5]$. If we apply a third-degree polynomial transformation, each value is expanded into $(1,x,x^2, x^3)$. This results in a new feature matrix:

$$
\begin{split}
X{'} = [& [ 1., &~~1., &~~~~1., &~~~~~1.], \\
        & [ 1., &~~2., &~~~~4., &~~~~~8.], \\
        & [ 1., &~~3., &~~~~9., &~~~27.], \\
        & [ 1., &~~4., &~~16., &~~~64.], \\
        & [ 1., &~~5., &~~25., &~125.]
       ]
\end{split}
$$

In [ ]:
x = np.array([1, 2, 3, 4, 5])
x.reshape(-1, 1)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

x = np.array([1, 2, 3, 4, 5])

poly = PolynomialFeatures(degree=3)
poly.fit_transform(x.reshape(-1, 1))

<br>

This transformed dataset can then be fed directly into a linear regression model, enabling it to learn non-linear patterns through these engineered polynomial features.

In [ ]:
errors = np.random.normal(0, 3, size=200)
x = np.linspace(0, 100, 200)
y = 30 + (-0.3 * x)+ (0.005 * x ** 2) + errors

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(x, y);

In [ ]:
# we take a small subsample of x
np.random.seed(46)

subset_indices = np.random.choice(np.arange(len(x)), size=20)
subset_indices.sort()
subset_indices

In [ ]:
x[subset_indices]

In [ ]:
temp = x[subset_indices].reshape(-1, 1)
print(temp.shape)
temp[0:5]

In [ ]:
poly = PolynomialFeatures(degree=2)
X_vals_transformed = poly.fit_transform(x[subset_indices].reshape(-1, 1))    
X_vals_transformed[0:5]

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_vals_transformed, y[subset_indices].reshape(-1, 1))

x_axis= np.arange(0, max(x)).reshape(-1, 1)
X_axis_transformed = poly.transform(x_axis)
y_hat = lin_reg.predict(X_axis_transformed)

plt.figure(figsize=(10, 5))
plt.scatter(x[subset_indices], y[subset_indices])
plt.plot(x_axis, y_hat, label="Polynomial degree %s" % 2)
plt.legend();

#### Increasing the Polynomial  Degree

When working with linear regression, recall that the goal is to find the model that minimizes the Residual Sum of Squares (RSS). In polynomial regression, even though we introduce non-linear terms in $x$, the model is still linear in its coefficients, so the same RSS minimization framework applies. By increasing the degree of the polynomial, we add flexibility to the model, allowing it to better capture complex patterns in the data. As a result, the training RSS will typically decrease as we move from a lower-degree polynomial (e.g., linear) to a higher-degree one. In the example above, we observe that using a higher-degree polynomial can improve the fit by more closely following the curvature of the data. 

In [ ]:
from sklearn.linear_model import LinearRegression

plt.figure(figsize=(15, 10))

x_axis= np.arange(0, max(x)).reshape(-1, 1)

for i, polDegree in enumerate([2, 5, 9, 25]):
    plt.subplot(2, 2, i+1)

    plt.scatter(x[subset_indices], y[subset_indices])
    poly = PolynomialFeatures(degree=polDegree)
    X_vals_transformed = poly.fit_transform(x[subset_indices].reshape(-1, 1))    
    X_axis_transformed = poly.transform(x_axis)

    lin_reg = LinearRegression()
    lin_reg.fit(X_vals_transformed, y[subset_indices].reshape(-1, 1))
    y_hat = lin_reg.predict(X_axis_transformed)

    plt.plot(x_axis[0:-4], y_hat[0:-4], label="Polynomial degree %s" % polDegree)
    plt.ylim(min(y[subset_indices]) - 5, max(y[subset_indices]) + 8)
    plt.legend(fontsize=18)

Although higher-degree polynomials often produce better fits to the data, this added flexibility can quickly become problematic. As the degree of the polynomial increases, the model gains the ability to bend and adapt more aggressively to the data. In fact, an $n^{th}$-degree polynomial can have up to $n−1$ turning points, allowing it to change direction multiple times and closely track the observed values.

However, this flexibility can lead to **overfitting**. High-degree polynomials tend to introduce unnecessary oscillations that are unlikely to reflect the true underlying relationship in the data. Instead, the model begins to capture random noise, resulting in a curve that fits the data extremely well but does not generalize to new, unseen observations. Even though such a model may pass through or near most data points, its predictive performance can be poor.

To address this issue, we can use a train-validation split to select an appropriate polynomial degree. By evaluating model performance on unseen validation data, we can identify the level of complexity that balances fit and generalization. This approach is widely applicable and forms the basis for model selection in many machine learning and statistical learning methods.